# Target Preparation

Assess a protein, review component decisions, prepare it, and find candidate binding pockets with one `TargetPrep` object.

This notebook uses blocking loops-off preparation. Loop modelling uses `start()`, `wait()`, and `get_results()` instead.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from deeporigin.drug_discovery import BRD_DATA_DIR, Protein, TargetPrep
from deeporigin.platform import DeepOriginClient

client = DeepOriginClient.from_disk()
client

## Configure the target

`TargetPrep` registers the protein when needed. Automatic pocket finding requires an explicit pocket count and minimum size; choose values appropriate for your target rather than relying on hidden defaults.

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")

prep = TargetPrep(
    protein=protein,
    pdb_id="1EBY",
    model_missing_loops=False,
    pocket_count=3,
    pocket_min_size=80,
    client=client,
)
prep

## Assess and review

`recommend()` blocks while it creates the source Structure Report and component inventory. The temporary recommendation execution is deliberately not attached to `prep`; `prep.id` remains unset for the later preparation execution.

In [ ]:
recommendation = prep.recommend()
prep.source_report

In [ ]:
reviews = recommendation(decision="review")
reviews

Resolve every `review` decision before preparation. This example skips all reviewed components; inspect the table and use `keep()` where appropriate for your target.

In [ ]:
prep.skip(reviews)
prep.selection

## Prepare and find pockets

With loop modelling disabled, `run()` blocks until protein preparation, the prepared Structure Report, and pocket finding finish. Use `start()` followed by `wait()` for loop modelling or a non-blocking workflow.

In [ ]:
result = prep.run()
result

`TargetPrepResult` keeps each child artifact optional so completed work remains accessible if a later step fails. The source report remains on `prep.source_report`; the aggregate contains artifacts from preparation.

In [ ]:
{
    "prepared_protein": result.prepared_protein,
    "prepared_report": result.prepared_report,
    "pocket_count": len(result.pockets or []),
    "extracted_ligands": result.extracted_ligands,
    "audit_file_path": result.audit_file_path,
    "selection_file_path": result.selection_file_path,
}